## NLP with Naive Bayes and PySpark: SMS Spam Dataset

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('nlp_nb').getOrCreate()

D:\miniconda3\envs\pyspark_env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [10]:
!powershell -command Get-Content SMSSpamCollection -TotalCount 5

ham	Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
ham	Ok lar... Joking wif u oni...
spam	Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
ham	U dun say so early hor... U c already then say...
ham	Nah I don't think he goes to usf, he lives around here though


In [11]:
df = spark.read.csv('SMSSpamCollection', inferSchema=True, sep='\t')

In [12]:
df.show()

+----+--------------------+
| _c0|                 _c1|
+----+--------------------+
| ham|Go until jurong p...|
| ham|Ok lar... Joking ...|
|spam|Free entry in 2 a...|
| ham|U dun say so earl...|
| ham|Nah I don't think...|
|spam|FreeMsg Hey there...|
| ham|Even my brother i...|
| ham|As per your reque...|
|spam|WINNER!! As a val...|
|spam|Had your mobile 1...|
| ham|I'm gonna be home...|
|spam|SIX chances to wi...|
|spam|URGENT! You have ...|
| ham|I've been searchi...|
| ham|I HAVE A DATE ON ...|
|spam|XXXMobileMovieClu...|
| ham|Oh k...i'm watchi...|
| ham|Eh u remember how...|
| ham|Fine if thats th...|
|spam|England v Macedon...|
+----+--------------------+
only showing top 20 rows


In [13]:
df = df.withColumnRenamed('_c0', 'class').withColumnRenamed('_c1', 'text')

#### Clean Data

In [14]:
from pyspark.sql.functions import length

In [15]:
df = df.withColumn('length', length(df['text']))

In [16]:
df.show()

+-----+--------------------+------+
|class|                text|length|
+-----+--------------------+------+
|  ham|Go until jurong p...|   111|
|  ham|Ok lar... Joking ...|    29|
| spam|Free entry in 2 a...|   155|
|  ham|U dun say so earl...|    49|
|  ham|Nah I don't think...|    61|
| spam|FreeMsg Hey there...|   147|
|  ham|Even my brother i...|    77|
|  ham|As per your reque...|   160|
| spam|WINNER!! As a val...|   157|
| spam|Had your mobile 1...|   154|
|  ham|I'm gonna be home...|   109|
| spam|SIX chances to wi...|   136|
| spam|URGENT! You have ...|   155|
|  ham|I've been searchi...|   196|
|  ham|I HAVE A DATE ON ...|    35|
| spam|XXXMobileMovieClu...|   149|
|  ham|Oh k...i'm watchi...|    26|
|  ham|Eh u remember how...|    81|
|  ham|Fine if thats th...|    56|
| spam|England v Macedon...|   155|
+-----+--------------------+------+
only showing top 20 rows


In [17]:
df.groupby('class').mean().show()

+-----+-----------------+
|class|      avg(length)|
+-----+-----------------+
|  ham|71.45431945307645|
| spam|138.6706827309237|
+-----+-----------------+



### Feature Transformation

#### Creating Assemblers / Features

In [19]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, StringIndexer, IDF

In [20]:
tokenizer = Tokenizer(inputCol='text', outputCol='token_text')
stop_word_remover = StopWordsRemover(inputCol='token_text', outputCol='stop_tokens')
count_vec = CountVectorizer(inputCol='stop_tokens', outputCol='c_vec')
idf = IDF(inputCol="c_vec", outputCol='tf_idf')
ham_spam_to_num = StringIndexer(inputCol='class', outputCol='label')

In [21]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.linalg import Vector

In [22]:
cleaned = VectorAssembler(inputCols=['tf_idf', 'length'], outputCol='features')

#### Naive Bayes Model

In [23]:
from pyspark.ml.classification import NaiveBayes

In [24]:
nb = NaiveBayes()

#### PipeLine

In [25]:
from pyspark.ml import Pipeline

In [26]:
pipeline = Pipeline(stages=[
    ham_spam_to_num,
    tokenizer,
    stop_word_remover,
    count_vec,
    idf,
    cleaned
])

In [27]:
cleaner = pipeline.fit(df)

In [28]:
clean_df = cleaner.transform(df)

In [29]:
clean_df.show()

+-----+--------------------+------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+
|class|                text|length|label|          token_text|         stop_tokens|               c_vec|              tf_idf|            features|
+-----+--------------------+------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+
|  ham|Go until jurong p...|   111|  0.0|[go, until, juron...|[go, jurong, poin...|(13423,[7,11,31,6...|(13423,[7,11,31,6...|(13424,[7,11,31,6...|
|  ham|Ok lar... Joking ...|    29|  0.0|[ok, lar..., joki...|[ok, lar..., joki...|(13423,[0,24,302,...|(13423,[0,24,302,...|(13424,[0,24,302,...|
| spam|Free entry in 2 a...|   155|  1.0|[free, entry, in,...|[free, entry, 2, ...|(13423,[2,13,19,3...|(13423,[2,13,19,3...|(13424,[2,13,19,3...|
|  ham|U dun say so earl...|    49|  0.0|[u, dun, say, so,...|[u, dun, say, ear...|(13423,[0,69,80,1...|(13423,[0,69,8

#### Train Model and Evaluation

In [30]:
clean_df = clean_df.select(['label', 'features'])

In [31]:
(train, test) = clean_df.randomSplit([0.7, 0.3], seed=42)

In [32]:
pred = nb.fit(train)

In [33]:
res = pred.transform(test)

In [34]:
res.show()

+-----+--------------------+--------------------+--------------------+----------+
|label|            features|       rawPrediction|         probability|prediction|
+-----+--------------------+--------------------+--------------------+----------+
|  0.0|(13424,[0,1,2,41,...|[-1061.7658581713...|[1.0,2.6778911707...|       0.0|
|  0.0|(13424,[0,1,5,20,...|[-804.30303759114...|[1.0,8.5979861672...|       0.0|
|  0.0|(13424,[0,1,7,8,1...|[-1170.0081128236...|[1.0,1.0611874952...|       0.0|
|  0.0|(13424,[0,1,7,15,...|[-657.32468708519...|[1.0,1.3859093519...|       0.0|
|  0.0|(13424,[0,1,12,33...|[-444.25194843761...|[1.0,1.4749998397...|       0.0|
|  0.0|(13424,[0,1,14,18...|[-1378.8974034449...|[1.0,1.2151759809...|       0.0|
|  0.0|(13424,[0,1,14,31...|[-216.49125506413...|[1.0,1.2461135199...|       0.0|
|  0.0|(13424,[0,1,18,20...|[-831.78489243553...|[1.0,2.9218828730...|       0.0|
|  0.0|(13424,[0,1,22,27...|[-774.67325076198...|[1.0,2.1522453133...|       0.0|
|  0.0|(13424,[0

In [35]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [36]:
evaluator = MulticlassClassificationEvaluator()
acc = evaluator.evaluate(res)
print(f"Accuracy: {acc * 100}%")

Accuracy: 92.59862204107951%
